# Práctica 2: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [Keras](https://keras.io/2.15/api/) de Python.

In [6]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

### Ejercicio 1

El hormigón es el material más importante en la ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de su edad y sus ingredientes.

El fichero `concrete_data.csv` contiene la siguiente información acerca de diferentes muestras de hormigón:

* Contenido de cemento (`Cement`).
* Contenido de escoria de alto horno (`Blast Furnace Slag`).
* Contenido de cenizas volantes (`Fly Ash`).
* Contenido de agua (`Water`).
* Contenido de superplastificantes (`Superplasticizer`).
* Contenido de agregados gruesos (`Coarse Aggregate`).
* Contenido de agregados finos (`Fine Aggregate`).
* Edad del hormigón (`Age`).

El objetivo es predecir la resistencia a la compresión (`Strength`) a partir de esos atributos continuos.

Se pide realizar lo siguiente:

1. Dividir el conjunto de datos en un subconjunto de entrenamiento y un subconjunto de prueba.
2. Abordar la tarea mediante una red neuronal que tenga una única capa oculta con 16 neuronas y función de activación sigmoide.
3. Entrenar la red durante 50 épocas, usando el error cuadrático medio como función de coste a minimizar.
4. Calcular el rendimiento de la red como el error absoluto medio sobre el conjunto de prueba.
5. Experimentar con distintas arquitecturas de la red (cantidad de capas ocultas, cantidad de neuronas en cada capa oculta, función de activación de las capas) e hiperparámetros de entrenamiento (factor de aprendizaje, tamaño de los minilotes, número de épocas), tratando de encontrar una red con un error absoluto medio sobre el conjunto de prueba menor a 0.1.

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

from keras.utils import set_random_seed

set_random_seed(5470)

import pandas as pd

data = pd.read_csv('concrete_data.csv')
data.head()



,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [10]:
# 1. Dividir en subconjunto de entrenamiento y prueba
from sklearn.model_selection import train_test_split
atributos = data.loc[:,'Cement':'Age']
objetivo = data.loc[:,'Strength']


(atributos_train, atributos_test, 
 objetivo_train, objetivo_test) = train_test_split(atributos, objetivo, test_size=.2)


In [12]:
# Antes de Abordar el punto 2. Normalizamos los datos.
from keras.layers import Normalization
import numpy as np

normalizador = Normalization()
normalizador.adapt(atributos_train.to_numpy())

np.mean(normalizador(atributos_train), axis=0)

array([-4.28001492e-08,  5.96046448e-08,  2.18453913e-08,  1.18883776e-07,
       -2.11220339e-08, -1.75183587e-07,  1.62321385e-07,  3.61678665e-09],
      dtype=float32)

In [ ]:
# Comprobar la normalización -> media y varianza con valores entre 0 y 1
np.var(normalizador(atributos_train), axis = 0)
np.me

array([1.0000007 , 1.0000027 , 1.0000027 , 0.9999988 , 0.99999726,
       1.0000011 , 0.99999994, 0.99999857], dtype=float32)

In [15]:
# 2. Abordar la tarea mediante una red neuronal que tenga una única capa oculta con 16 neuronas
#  y función de activación sigmoide.

from keras import Sequential, Input
from keras.layers import Dense


neuronal_network = Sequential()
neuronal_network.add(Input(shape=(8,)))
neuronal_network.add(normalizador)
neuronal_network.add(Dense(16,activation='sigmoid'))
neuronal_network.add(Dense(1))

neuronal_network.summary()


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_3 (Normalization) │ (None, 8)              │            17 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 16)             │           144 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 178 (716.00 B)

 Trainable params: 161 (644.00 B)

 Non-trainable params: 17 (72.00 B)

In [16]:
# 3. Entrenar la red durante 50 épocas, usando el error cuadrático medio como función de coste a minimizar.

neuronal_network.compile(optimizer='SGD',loss='mean_squared_error', metrics= ['mean_absolute_error'])
neuronal_network.fit(atributos_train, objetivo_train, batch_size=256, epochs=50)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 1355.3864 - mean_absolute_error: 32.4850 
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 633.0291 - mean_absolute_error: 20.2345 
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 285.1031 - mean_absolute_error: 13.1621 
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 182.8779 - mean_absolute_error: 10.6471
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 146.9384 - mean_absolute_error: 9.6658  
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 128.4306 - mean_absolute_error: 9.1024 
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 117.6032 - mean_absolute_error: 8.7110 
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 110.7386 - mean_absolute_error: 8.4205 
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 105.9995 - mean_absolute_error: 8.1978 
Epoch 10/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 102.4558 - mean_absolute_error: 8.0095 
Epoch 11/50
4/4 ━━━━━━━━━━━━━━━━

In [17]:
# 4. Calcular el rendimiento de la red como el error absoluto medio sobre el conjunto de prueba.

loss, mae = neuronal_network.evaluate(atributos_test, objetivo_test)
print("MSE (loss):", loss)
print("MAE:", mae)

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 64.0627 - mean_absolute_error: 6.2642  
MSE (loss): 64.06268310546875
MAE: 6.264230251312256


In [18]:
# 5. Experimentar con distintas arquitecturas de la red (cantidad de capas ocultas, cantidad de neuronas en cada capa 
# oculta, función de activación de las capas) e hiperparámetros de entrenamiento 
# (factor de aprendizaje, tamaño de los minilotes, número de épocas), 
# tratando de encontrar una red con un error absoluto medio sobre el conjunto de prueba menor a 0.1.

# cambio en el factor de aprendizaje
from keras.optimizers import SGD

neuronal_network = Sequential()
neuronal_network.add(Input(shape=(8,)))
neuronal_network.add(normalizador)
neuronal_network.add(Dense(32,activation='sigmoid'))
neuronal_network.add(Dense(16,activation='sigmoid'))
neuronal_network.add(Dense(16,activation='sigmoid'))
neuronal_network.add(Dense(16,activation='sigmoid'))
neuronal_network.add(Dense(1))

compile = neuronal_network.compile(
    optimizer=SGD(learning_rate=0.001),
    loss='mean_squared_error',
    metrics= ['mean_absolute_error'])
neuronal_network.fit(atributos_train, objetivo_train, batch_size=256, epochs=100)


Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1543.8768 - mean_absolute_error: 35.4853  
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 1428.8527 - mean_absolute_error: 33.8261 
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1315.5834 - mean_absolute_error: 32.1137 
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1201.7607 - mean_absolute_error: 30.3017
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1088.4839 - mean_absolute_error: 28.4064 
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 978.6597 - mean_absolute_error: 26.4938  
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 875.4844 - mean_absolute_error: 24.6429  
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 781.4421 - mean_absolute_error: 22.9271 
Epoch 9/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 697.9365 - mean_absolute_error: 21.3835 
Epoch 10/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 625.3526 - mean_absolute_error: 20.0408 
Epoch 11/10

### Ejercicio 2

El fichero `cars.csv` contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos discretos:

* Precio de compra (`buying`): posibles valores `vhigh`, `high`, `med`, `low`.
* Coste de mantenimiento (`maint`): posibles valores `vhigh`, `high`, `med`, `low`.
* Número de puertas (`doors`): posibles valores `2`, `3`, `4`, `5more`.
* Número de asientos (`persons`): posibles valores `2`, `4`, `more`.
* Tamaño del maletero (`lug_boot`): posibles valores `small`, `med`, `big`.
* Nivel de seguridad estimada (`safety`): posibles valores `low`, `med`, `high`.

La idoneidad de cada coche se indica mediante el atributo `acceptability`, que los clasifica como `unacc`, `acc`, `good` o `vgood`.

Se pide realizar lo siguiente:

1. Dividir el conjunto de datos en un subconjunto de entrenamiento y un subconjunto de prueba.
2. Abordar la tarea mediante una red neuronal que tenga una única capa oculta con 16 neuronas y función de activación tangente hiperbólica.
3. Entrenar la red durante 50 épocas, usando la entropía cruzada categórica como función de coste a minimizar.
4. Calcular el rendimiento de la red como la tasa de acierto sobre el conjunto de prueba.
5. Experimentar con distintas arquitecturas de la red (cantidad de capas ocultas, cantidad de neuronas en cada capa oculta, función de activación de las capas) e hiperparámetros de entrenamiento (factor de aprendizaje, tamaño de los minilotes), tratando de encontrar una red que, entrenada únicamente durante 50 épocas, tenga una tasa de acierto sobre el conjunto de prueba superior a 0.9.

In [19]:
# leemos los datos
import pandas as pf

datos = pd.read_csv('cars.csv')
# mostramos las primeras lineas
datos.head()

,buying,maint,doors,persons,lug_boot,safety,acceptability
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [ ]:
# 1. Dividir el conjunto de datos en un subconjunto de entrenamiento y un subconjunto de prueba.
atributos_entrenamiento = datos.loc[:,'buying':'safety']
atributos_objetivo = datos.loc[:,'acceptability':'acceptability']
atributos_objetivo.head()

InvalidIndexError: (slice(None, None, None), slice('acceptability', 'acceptability', None))

### Ejercicio 3

Los púlsares son un tipo raro de estrella de neutrones que produce emisiones de radio detectables aquí en la Tierra. Son de considerable interés científico como sondas del espacio-tiempo, el medio interestelar y los estados de la materia.

A medida que los púlsares giran, su haz de emisión recorre el cielo y, cuando cruza nuestra línea de visión, produce un patrón detectable de emisión de radio de banda ancha. Como los púlsares giran rápidamente, este patrón se repite periódicamente. Por tanto, la búsqueda de púlsares implica buscar señales de radio periódicas con grandes radiotelescopios.

Cada púlsar produce un patrón de emisión algo diferente, que varía levemente con cada rotación. Por lo tanto, una detección de señal potencial conocida como «candidata» se promedia a lo largo de muchas rotaciones del púlsar, según lo determinado por la duración de una observación. A falta de información adicional, cada candidato podría describir un púlsar real. Sin embargo, en la práctica, casi todas las detecciones son causadas por interferencias de radiofrecuencia (RFI) y ruido, lo que dificulta encontrar señales legítimas.

El fichero `pulsar_stars.csv` contiene datos acerca de una serie de púlsares reales y de ejemplos espurios producidos por RFI y ruido. Cada candidato se describe mediante ocho atributos continuos extraídos de las señales recibidas.

Se pide construir una red neuronal que permita abordar, con el mayor rendimiento posible, la tarea de de determinar si un candidato es o no un púlsar.

### Ejercicio 4

Los [abulones](https://es.wikipedia.org/wiki/Haliotis) son una familia de moluscos gasterópodos. La edad de cada individuo está correlacionada con el número de anillos de su concha y, por tanto, puede determinarse cortando la concha a través del cono, tiñéndola y contando el número de anillos a través de un microscopio. Este procedimiento requiere mucho tiempo y es propenso a errores, por lo que sería preferible poder determinar la edad directamente a partir de medidas físicas más fáciles de obtener.

El fichero `abalone.csv` contiene la siguiente información de distintos individuos de abulones:

* Sexo (`Sex`): atributo discreto con posibles valores `M` (macho), `F` (hembra) e `I` (infante).
* Longitud (`Length`) en milímetros.
* Diámetro (`Diameter`) en milímetros.
* Altura (`Height`) en milímetros.
* Peso total (`Whole_weight`) en gramos.
* Peso sin la concha (`Shucked_weight`) en gramos.
* Peso intestinal (`Viscera_weight`) en gramos.
* Peso de la concha (`Shell_weight`) en gramos.

Se pide construir una red neuronal que permita abordar, con el mayor rendimiento posible, la tarea de predecir el número de anillos (`Rings`) a partir de los atributos anteriores (entonces bastaría sumar 1.5 a ese número de anillos para obtener la edad, en años, del individuo).